In [ ]:
import json
import pandas as pd

path = "forken.json"


with open(path, "r", encoding="utf-8") as f:
    data = json.load(f)  # dict: { "NN-NNN": [ {enrollment}, ... ], ... }

rows = []
for protocol_number, enrollments in data.items():
    # Normalize to a list of dicts
    if isinstance(enrollments, dict):
        enrollments = [enrollments]
    if not isinstance(enrollments, list):
        continue
    for rec in enrollments:
        if isinstance(rec, dict):
            row = rec.copy()
            row["protocol_number"] = protocol_number
            rows.append(row)

df = pd.DataFrame(rows)

# no non-DFCI consents
df = df[~df.studySiteOriginalMrn.isnull()]
df = df[df['studySiteOriginalMrn'].str.contains('DFCI')]

# Parse dfci_mrn from studySiteOriginalMrn by removing leading "DFCI"
if "studySiteOriginalMrn" in df.columns:
    df["dfci_mrn"] = (
        df["studySiteOriginalMrn"]
        .astype("string")
        .str.replace(r"(?i)^\s*DFCI\s*", "", regex=True)
        .str.strip()
    )
else:
    df["dfci_mrn"] = pd.NA

df['trial_start_dt'] = pd.to_datetime(df['consentDate'])

print(df.head())

In [ ]:
df.info()

In [ ]:
df.diseaseSite.value_counts().head(25)

In [ ]:
df = df[['dfci_mrn','protocol_number','trial_start_dt']]
df.to_csv('true_dfci_enrollments.csv')